# Movie Match - Recomendador por embeddings

Dos o más personas eligen películas que les gustan. El sistema genera un **vector de gusto** para cada persona usando embeddings semánticos de las sinopsis (`overview`) obtenidos en vivo de la API de **TMDB**, calcula el **gusto combinado** (punto medio en el espacio vectorial) y recomienda películas del catálogo que maximicen la similitud coseno.

**Pipeline:**
1. Buscar películas por título en TMDB -> obtener `overview`
2. Generar embeddings de las sinopsis usando `sentence-transformers` (`all-MiniLM-L6-v2`)
3. Promediar los embeddings por persona -> vector de gusto individual
4. Calcular el punto medio de los vectores -> vector de gusto combinado
5. Obtener candidatas (Populares + Top Rated en TMDB)
6. Similitud coseno entre el vector combinado y el catálogo -> Ranking Top N
7. Visualización interactiva/2D con PCA

## 0. Instalación de dependencias

In [ ]:
%pip install -r requirements.txt -q

## 1. Configuración y Credenciales

Crea un archivo `.env` en la raíz del proyecto basándote en `.env.example`:
```
TMDB_BEARER_TOKEN=tu_token_aca
```

In [ ]:
from __future__ import annotations
import os
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TMDB_BEARER_TOKEN = os.getenv("TMDB_BEARER_TOKEN")

if not TMDB_BEARER_TOKEN:
    raise ValueError(
        "[ERROR] No se encontró TMDB_BEARER_TOKEN. Crea un archivo .env en esta carpeta "
        "con la línea: TMDB_BEARER_TOKEN=tu_token_aca"
    )

TMDB_BASE = "https://api.themoviedb.org/3"
HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_BEARER_TOKEN}",
}

def _get(endpoint, params=None):
    params = params or {}
    params.setdefault("language", "es-AR")
    r = requests.get(f"{TMDB_BASE}{endpoint}", params=params, headers=HEADERS)
    r.raise_for_status()
    return r.json()

## 2. Búsqueda de películas en TMDB

In [ ]:
def search_movie(title: str):
    """Busca una película por título y devuelve datos básicos + overview."""
    data = _get("/search/movie", {"query": title})
    results = data.get("results", [])
    if not results:
        print(f"[WARNING] No encontré resultados para: {title!r}")
        return None
    top = results[0]
    return {
        "id": top["id"],
        "title": top["title"],
        "overview": top.get("overview", ""),
        "release_date": top.get("release_date", ""),
        "vote_average": top.get("vote_average"),
        "poster_path": top.get("poster_path"),
    }

## 3. Cargar Modelo de Embeddings

Usamos `all-MiniLM-L6-v2` de HuggingFace `sentence-transformers`.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_text(text: str) -> np.ndarray:
    return model.encode(text, normalize_embeddings=True)

## 4. Generación del Vector de Gusto

In [ ]:
def build_taste_vector(movie_titles: list[str]):
    found = []
    vectors = []
    for title in movie_titles:
        movie = search_movie(title)
        if movie and movie["overview"]:
            found.append(movie)
            vectors.append(embed_text(movie["overview"]))
    if not vectors:
        raise ValueError("No se pudo generar embedding para ninguna película de la lista.")
    taste_vector = np.mean(vectors, axis=0)
    taste_vector /= np.linalg.norm(taste_vector)
    return taste_vector, found

## 5. Ingreso de películas por Persona

In [ ]:
persona_1 = [
    "Interstellar",
    "Eternal Sunshine of the Spotless Mind",
    "Whiplash",
]

persona_2 = [
    "Coco",
    "La La Land",
    "Amelie",
]

vector_1, pelis_1 = build_taste_vector(persona_1)
vector_2, pelis_2 = build_taste_vector(persona_2)

vector_combinado = (vector_1 + vector_2) / 2
vector_combinado /= np.linalg.norm(vector_combinado)

print("Persona 1:", [p["title"] for p in pelis_1])
print("Persona 2:", [p["title"] for p in pelis_2])

## 6. Pool de Candidatas

In [ ]:
def get_candidate_pool(n_pages: int = 5) -> list[dict]:
    candidates = {}
    for endpoint in ["/movie/popular", "/movie/top_rated"]:
        for page in range(1, n_pages + 1):
            data = _get(endpoint, {"page": page})
            for m in data.get("results", []):
                if m.get("overview"):
                    candidates[m["id"]] = {
                        "id": m["id"],
                        "title": m["title"],
                        "overview": m["overview"],
                        "vote_average": m.get("vote_average"),
                        "poster_path": m.get("poster_path"),
                    }
    return list(candidates.values())

pool = get_candidate_pool(n_pages=5)
print(f"Pool de candidatas: {len(pool)} películas")

## 7. Similitud Coseno y Ranking

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ya_vistas = {p["id"] for p in pelis_1 + pelis_2}
pool_filtrado = [m for m in pool if m["id"] not in ya_vistas]

pool_embeddings = np.array([embed_text(m["overview"]) for m in pool_filtrado])
sims = cosine_similarity(vector_combinado.reshape(1, -1), pool_embeddings)[0]

for m, s in zip(pool_filtrado, sims):
    m["score"] = float(s)

ranking = sorted(pool_filtrado, key=lambda m: m["score"], reverse=True)
top_n = ranking[:10]

df_top = pd.DataFrame(top_n)[["title", "score", "vote_average"]]
df_top["score"] = df_top["score"].round(3)
df_top

## 8. Visualización PCA (2D)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

vecs_1 = np.array([embed_text(p["overview"]) for p in pelis_1])
vecs_2 = np.array([embed_text(p["overview"]) for p in pelis_2])
vecs_top = np.array([embed_text(m["overview"]) for m in top_n])

all_vecs = np.vstack([vecs_1, vecs_2, vector_combinado.reshape(1, -1), vecs_top])
pca = PCA(n_components=2)
coords = pca.fit_transform(all_vecs)

n1, n2, ntop = len(vecs_1), len(vecs_2), len(vecs_top)
c1 = coords[:n1]
c2 = coords[n1:n1+n2]
ccomb = coords[n1+n2:n1+n2+1]
ctop = coords[n1+n2+1:]

plt.figure(figsize=(9, 7))
plt.scatter(c1[:, 0], c1[:, 1], c="#4C72B0", s=100, label="Persona 1")
plt.scatter(c2[:, 0], c2[:, 1], c="#DD8452", s=100, label="Persona 2")
plt.scatter(ccomb[:, 0], ccomb[:, 1], c="black", marker="X", s=200, label="Gusto combinado")
plt.scatter(ctop[:, 0], ctop[:, 1], c="#55A868", s=60, alpha=0.6, label="Top 10 recomendadas")

for i, p in enumerate(pelis_1):
    plt.annotate(p["title"], c1[i], fontsize=8)
for i, p in enumerate(pelis_2):
    plt.annotate(p["title"], c2[i], fontsize=8)
plt.annotate(top_n[0]["title"], ctop[0], fontsize=9, weight="bold")

plt.legend()
plt.title("Mapa de gustos en el espacio de embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()